In [3]:
import pandas as pd
df = pd.read_csv(r"D:\development\aiml\tasks\part_33_supervised_ml_6\predictive_maintenance.csv")

In [4]:
import pandas as pd
import logging
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

logging.basicConfig(
    level = logging.INFO,
    format = "[+%(relativeCreated)dms] %(levelname)s: %(message)s"
)

logger = logging.getLogger(__name__)
log = logger.info

config = {
    "filepath": "predictive_maintenance.csv",
    "random_state": 42,
    "test_size": 0.2,
    "n_splits": 5,
    "cost_missed_failure": 10,
    "cost_false_alarm": 1,
    "xgb_params": {
        "n_estimators": 200,
        "max_depth": 5,
        "learning_rate": 0.05
    },
    "artificat_path" : "machine_failure_artifact.joblib",
    "target_columns": "Target",
    "required_columns": ['UDI', 'Product ID', 'Type', 'air_temperature',
       'process_temperature', 'rotational_speed', 'torque',
       'tool_wear', 'Target', 'Failure Type'],
    "categorical_columns": ["Type"],
    "numerical_columns": ["air_temperature", "process_temperature", "rotational_speed", "torque", "tool_wear"],
    "drop_columns": ["UDI", "Product ID", "Failure Type"]
}

COLUMN_MAPPING = {
    "Air temperature [K]": "air_temperature",
    "Process temperature [K]": "process_temperature",
    "Rotational speed [rpm]": "rotational_speed",
    "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear"
}

class DataValidationError(Exception):
    """Raise this when there is a mismatch of columns and stuff"""
    pass

def load_dataset(config: dict) -> pd.DataFrame:
    if "filepath" not in config:
        raise DataValidationError("Filepath doesnt contain dataset!! Please fix and rerun.")
    df = pd.read_csv(config["filepath"])

    df = df.rename(columns=COLUMN_MAPPING)

    for col in config["required_columns"]:
        if col not in df.columns:
            raise DataValidationError(f"Dataset doesnt contains {col}, make sure u have the correct dataset")


    positive_rate = df[config["target_columns"]].mean() * 100
    log(f"Dataset has been loaded \n Shape: {df.shape} \nPositive Rate:{positive_rate:.2f}%")
    return df
    # shape (10,000, 10)
    # positive rate: 3.39%

def cleaning_dataset(df: pd.DataFrame, config: dict):
    X = df.drop(columns=config["drop_columns"] + [config["target_columns"]])
    y = df[config["target_columns"]]

    #UDI and Product ID, is of no use in evalaution and training.
    # Target is dropped becuase it is the target column for y, and Failure Type is a leakage column.

    return X, y
def build_pipeline(config: dict, scale_pos_weight: float = 1.0) -> Pipeline:
    encoding_preprocessor = ColumnTransformer(
            transformers=[
                ("num", "passthrough", config["numerical_columns"]),
                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), config["categorical_columns"])
            ]
    )

    model = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=config["random_state"],
        **config["xgb_params"]
    )

    return Pipeline(
        steps=[
            ("encoding", encoding_preprocessor),
            ("classifier", model)
        ])

    # this function returns a pipeline that does encoding on numerical and categorical columns, and XGBoost model has been fitted.
    # 



SyntaxError: expected argument value expression (3621056769.py, line 75)

In [ ]:
from xgboost import XGBClassifier